### Taking a peek & sanity checks, one 3D image

Run this notebook on a few different wells in each batch, to get a sense for inter-well variability.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import ndimage

from functions import load_channel_stack

base_path = '/home/jdweiss1/orcd/scratch/15/287-tri'

channels = {
    1: 'DAPI',
    2: 'p-c-Jun (FITC)',
    3: 'ki67 (TRITC)',
    4: 'Mac cell trace (Cy5)',
    5: 'EpCAM (Cy7)',
}

First, let's check the distribution of fluorescence intensity by channel.

In [2]:
well_id = 'B02'

_, axs = plt.subplots(1, len(channels), figsize=(16, 4))

for ax, (channel_id, channel) in zip(axs, channels.items()):

    stack = load_channel_stack(base_path, well_id, channel_id)
    print(f'loading channel {channel} with shape {stack.shape}')

    pixels = np.ravel(stack)
    p_low, p_high = np.percentile(pixels, [0.1, 99.9])
    pixels_clipped = pixels[(pixels >= p_low) & (pixels <= p_high)]

    ax.hist(pixels_clipped, bins=100)
    ax.set_title(channel)

    del stack, pixels, pixels_clipped

plt.title('tri')
plt.savefig('figs/tri_histogram.png')

loading channel DAPI with shape (3, 2048, 2048)
loading channel p-c-Jun (FITC) with shape (3, 2048, 2048)
loading channel ki67 (TRITC) with shape (3, 2048, 2048)
loading channel Mac cell trace (Cy5) with shape (3, 2048, 2048)
loading channel EpCAM (Cy7) with shape (3, 2048, 2048)


Now, let's check for vignetting (bright center, dim edges).

In [3]:
# max intensity projection
dapi = load_channel_stack(base_path, well_id, 1)
mip = dapi.max(axis=0)

print(f'loading MIP for channel DAPI with shape {mip.shape}')

flatfield_estimate = ndimage.uniform_filter(mip.astype(float), size=mip.shape[0]//4)

plt.figure()
plt.imshow(flatfield_estimate, cmap='hot')
plt.savefig('figs/flatfield_estimate.png')

loading MIP for channel DAPI with shape (2048, 2048)


Now, let's check for channel bleed-through, e.g., highly correlated signal between channels.

In [4]:
mips = {}
for channel_id, channel in channels.items():
    stack = load_channel_stack(base_path, well_id, channel_id)
    mips[channel] = stack.max(axis=0).ravel()
    del stack

df = pd.DataFrame(mips)
print(df.sample(100_000).corr())  # sample for speed

                          DAPI  p-c-Jun (FITC)  ki67 (TRITC)  \
DAPI                  1.000000        0.320414      0.561062   
p-c-Jun (FITC)        0.320414        1.000000      0.228726   
ki67 (TRITC)          0.561062        0.228726      1.000000   
Mac cell trace (Cy5)  0.108886        0.112799      0.101641   
EpCAM (Cy7)           0.572332        0.240045      0.348670   

                      Mac cell trace (Cy5)  EpCAM (Cy7)  
DAPI                              0.108886     0.572332  
p-c-Jun (FITC)                    0.112799     0.240045  
ki67 (TRITC)                      0.101641     0.348670  
Mac cell trace (Cy5)              1.000000     0.051801  
EpCAM (Cy7)                       0.051801     1.000000  
